In [1]:
import scanpy as sc
import pandas as pd
import sys
import os

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# =====================================================
# 1. 参数设置（只需改这里）
# =====================================================
BASE_DIR = r"D:\Scunpair_Project\Benchmark_result(strong_5repeat)"
DATASET = "D31"
IS_MATCH = "unpaired"

METHOD = [
    "UniDISA",
    "MaxFuse",
    "scConfluence",
    "BindSC",
    "GLUE",
    "Seurat",
    "Uniport"
]

N_REPEAT = 5

# 自定义指标
sys.path.append(r"D:\Scunpair_Project\Diagonal-integration")
import mycode

# =====================================================
# 2. 创建 DATASET 结果目录
# =====================================================
RESULT_DIR = os.path.join(BASE_DIR, DATASET)
os.makedirs(RESULT_DIR, exist_ok=True)

# =====================================================
# 3. Label transfer（Accuracy / F1）
# =====================================================
label_transfer_xlsx = os.path.join(RESULT_DIR, "label_transfer.xlsx")

with pd.ExcelWriter(label_transfer_xlsx, engine="xlsxwriter") as writer:

    for r in range(1, N_REPEAT + 1):
        print(f"[Label transfer] repeat {r}")

        acc_list = []
        f1_list = []

        for method in METHOD:
            file_path = os.path.join(
                BASE_DIR,
                method,
                IS_MATCH,
                DATASET,
                f"repeat{r}",
                "coembed.h5ad"
            )

            if not os.path.exists(file_path):
                raise FileNotFoundError(file_path)

            adata = sc.read_h5ad(file_path)

            acc, f1 = mycode.metrics.calculate_shared_type_transfer_metrics(
                adata,
                batch_key="domain",
                label_key="cell_type",
                embed="X_emb"
            )

            acc_list.append(acc)
            f1_list.append(f1)

        df = pd.DataFrame({
            "Method": METHOD,
            "Accuracy": acc_list,
            "F1_Score": f1_list
        })

        df.to_excel(
            writer,
            sheet_name=f"repeat_{r}",
            index=False
        )

print(f"✅ Saved: {label_transfer_xlsx}")

c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Label transfer] repeat 1
[Label transfer] repeat 2
[Label transfer] repeat 3
[Label transfer] repeat 4
[Label transfer] repeat 5
✅ Saved: D:\Scunpair_Project\Benchmark_result(strong_5repeat)\D31\label_transfer.xlsx


In [2]:
bio_bc_xlsx = os.path.join(RESULT_DIR, "bio_bc.xlsx")

with pd.ExcelWriter(bio_bc_xlsx, engine="xlsxwriter") as writer:

    for r in range(1, N_REPEAT + 1):
        print(f"[Bio / BC] repeat {r}")

        adata = None

        for method in METHOD:
            file_path = os.path.join(
                BASE_DIR,
                method,
                IS_MATCH,
                DATASET,
                f"repeat{r}",
                "coembed.h5ad"
            )

            if not os.path.exists(file_path):
                raise FileNotFoundError(file_path)

            coembed = sc.read_h5ad(file_path)

            if adata is None:
                adata = coembed.copy()

            # 每个方法的 embedding
            adata.obsm[method] = coembed.X

        bm = Benchmarker(
            adata,
            batch_key="domain",
            label_key="cell_type",
            bio_conservation_metrics=BioConservation(
                silhouette_label=False
            ),
            batch_correction_metrics=BatchCorrection(
                pcr_comparison=False
            ),
            embedding_obsm_keys=METHOD,
            n_jobs=7,
        )

        bm.benchmark()
        results = bm.get_results()

        results.to_excel(
            writer,
            sheet_name=f"repeat_{r}"
        )

print(f"✅ Saved: {bio_bc_xlsx}")

c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\scanpy\preprocessing\_pca\__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


[Bio / BC] repeat 1


Embeddings:   0%|          | 0/7 [00:00<?, ?it/s]

INFO     ASDC consists of a single batch or is too small. Skip.                                                    
INFO     B intermediate consists of a single batch or is too small. Skip.                                          
INFO     B.Activated consists of a single batch or is too small. Skip.                                             


c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\scib_metrics\metrics\_kbet.py:182: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  idx_nonan = np.flatnonzero(np.in1d(labs, comp_size[comp_size >= comp_size_thresh].index))
c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\scib_metrics\metrics\_kbet.py:182: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  idx_nonan = np.flatnonzero(np.in1d(labs, comp_size[comp_size >= comp_size_thresh].index))


...

INFO     CD8 TCM consists of a single batch or is too small. Skip.                                                 
INFO     CD8 TEM consists of a single batch or is too small. Skip.                                                 
INFO     Eryth consists of a single batch or is too small. Skip.                                                   
INFO     HSPC consists of a single batch or is too small. Skip.                                                    
INFO     MAIT consists of a single batch or is too small. Skip.                                                    
INFO     Mono.CD16 consists of a single batch or is too small. Skip.                                               


c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\scib_metrics\metrics\_kbet.py:182: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  idx_nonan = np.flatnonzero(np.in1d(labs, comp_size[comp_size >= comp_size_thresh].index))


INFO     Platelets consists of a single batch or is too small. Skip.                                               
INFO     T.CD4.Memory consists of a single batch or is too small. Skip.                                            
INFO     T.CD8.Effector consists of a single batch or is too small. Skip.                                          
INFO     Treg consists of a single batch or is too small. Skip.                                                    
INFO     cDC2 consists of a single batch or is too small. Skip.                                                    
INFO     gdT consists of a single batch or is too small. Skip.                                                     
INFO     pDC consists of a single batch or is too small. Skip.                                                     


c:\Users\Administrator\miniconda3\envs\mycode\lib\site-packages\scib_metrics\metrics\_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)
Embeddings: 100%|██████████| 7/7 [00:57<00:00,  8.26s/it]


✅ Saved: D:\Scunpair_Project\Benchmark_result(strong_5repeat)\D31\bio_bc.xlsx
